# 🏛️ Sorai Tokyo — Sorai Index（都心高級賃貸 住環境・静粛性スコアリング）

> **Brand Philosophy**: *Quiet Luxury（静かな贅沢）× お財布防衛（地盤安心・適正相場）*
>
> 国土交通省の「不動産情報ライブラリ」一次データを活用し、既存ポータル（SUUMO等）の「駅徒歩・築年数・家賃」という記号的スペックを超えた、**真の住環境クオリティ（静粛性・地盤安心・相場適正度）**を客観的にスコアリングします。

---

## 1. 環境構築 ＆ 必要ライブラリのインストール
Google Colab Pro のハイメモリ環境（RAM 50GB+）向けに、GIS空間演算・地図可視化ライブラリをセットアップします。

In [ ]:
# 必要なGIS・可視化ライブラリのインストール
!pip install --quiet geopandas shapely folium plotly requests rtree pandas pyproj

import os
import math
import json
import time
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, shape, mapping
import folium
from folium import plugins

print("✅ Geospatial libraries loaded successfully.")

## 2. API 認証設定（不動産情報ライブラリ）
Colab の「シークレット（鍵アイコン）」機能から `LIBRARY_API_KEY` を取得します。
※ ノートブック内に直接 API キーをハードコードしない安全設計です。

In [ ]:
# Colab Secrets または環境変数から API キーを取得
try:
    from google.colab import userdata
    API_KEY = userdata.get('LIBRARY_API_KEY')
except Exception:
    API_KEY = os.environ.get('LIBRARY_API_KEY', '')

if not API_KEY:
    print("⚠️ [注意] LIBRARY_API_KEY が設定されていません。Colab左メニューの『鍵アイコン（Secrets）』に LIBRARY_API_KEY を登録してください。")
else:
    print("✅ LIBRARY_API_KEY detected.")

BASE_URL = "https://www.reinfolib.mlit.go.jp/ex-api/external"
ZOOM = 15

## 3. タイル座標変換 ＆ MLIT API キャッシュクライアント
Web メルカトル（ズーム 15）のタイル座標への高速変換と、同一タイルの再取得を防ぐローカルキャッシュ機構を定義します。

In [ ]:
def latlon_to_tile(lat: float, lon: float, zoom: int = 15):
    """緯度経度からタイル座標 (x, y) を算出"""
    n = 2 ** zoom
    lat_rad = math.radians(lat)
    x = math.floor(n * ((lon + 180) / 360))
    y = math.floor(n * (1 - (math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi)) / 2)
    return x, y

# キャッシュディレクトリ作成
CACHE_DIR = Path("./mlit_cache")
CACHE_DIR.mkdir(exist_ok=True)

def fetch_mlit_geojson(endpoint_code: str, x: int, y: int, z: int = 15) -> dict:
    """国交省APIからGeoJSONを取得（ローカルキャッシュ付き）"""
    cache_path = CACHE_DIR / f"{endpoint_code}_{z}_{x}_{y}.geojson"
    if cache_path.exists():
        with open(cache_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    
    if not API_KEY:
        return {"type": "FeatureCollection", "features": []}
    
    url = f"{BASE_URL}/{endpoint_code}"
    headers = {
        "Ocp-Apim-Subscription-Key": API_KEY,
        "Accept": "*/*"
    }
    params = {
        "response_format": "geojson",
        "z": z,
        "x": x,
        "y": y
    }
    
    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            with open(cache_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False)
            return data
        else:
            return {"type": "FeatureCollection", "features": []}
    except Exception as e:
        print(f"API取得エラー ({endpoint_code}): {e}")
        return {"type": "FeatureCollection", "features": []}

## 4. 空間結合（Spatial Join）＆ スコアリングエンジン

### スコア配点体系（合計 100 点満点）
1. **静粛性・住環境（Tranquility: 35点）**: 用途地域（第一種住居系は高得点、商業・準工業は減点）
2. **安全性・地盤（Safety: 35点）**: 洪水浸水想定、高潮、液状化リスクの有無
3. **相場適正度（Fair Value: 20点）**: 周辺取引価格との整合性
4. **生活スマート（Livability: 10点）**: 基本利便性

In [ ]:
# 用途地域コードと静粛性スコアの定義 (35点満点 / 漢数字・全角数字両対応)
ZONING_TRANQUILITY_SCORES = {
    "第一種低層住居専用地域": 35, "第１種低層住居専用地域": 35,
    "第二種低層住居専用地域": 33, "第２種低層住居専用地域": 33,
    "第一種中高層住居専用地域": 31, "第１種中高層住居専用地域": 31,
    "第二種中高層住居専用地域": 28, "第２種中高層住居専用地域": 28,
    "第一種住居地域": 26, "第１種住居地域": 26,
    "第二種住居地域": 22, "第２種住居地域": 22,
    "準住居地域": 19,
    "近隣商業地域": 14,
    "商業地域": 10,
    "準工業地域": 12,
    "工業地域": 5,
    "工業専用地域": 0
}

def evaluate_sorai_index(lat: float, lon: float, property_name: str = "対象物件") -> dict:
    """指定座標の Sorai Index を総合判定"""
    x, y = latlon_to_tile(lat, lon, ZOOM)
    pt = Point(lon, lat)
    
    # --- 1. 静粛性評価 (用途地域: XKT002) ---
    zoning_geojson = fetch_mlit_geojson("XKT002", x, y, ZOOM)
    detected_zone = "商業地域"
    tranquility_score = 15
    
    if zoning_geojson.get("features"):
        for feat in zoning_geojson["features"]:
            poly = shape(feat["geometry"])
            if poly.contains(pt):
                props = feat.get("properties", {})
                detected_zone = props.get("use_area_ja") or props.get("use_district_name") or props.get("用途地域", "商業地域")
                tranquility_score = ZONING_TRANQUILITY_SCORES.get(detected_zone, 15)
                break
    
    # --- 2. 安全性・地盤評価 (洪水浸水想定: XKT013, 液状化: XKT012) ---
    hazard_score = 35
    hazard_notes = []
    
    # 洪水浸水想定
    flood_geojson = fetch_mlit_geojson("XKT013", x, y, ZOOM)
    flood_depth = "0m (浸水想定区域外)"
    if flood_geojson.get("features"):
        for feat in flood_geojson["features"]:
            poly = shape(feat["geometry"])
            if poly.contains(pt):
                depth_val = feat.get("properties", {}).get("depth", "浸水域")
                flood_depth = f"{depth_val}"
                hazard_score -= 15
                hazard_notes.append(f"洪水浸水想定: {flood_depth}")
                break
    if not hazard_notes:
        hazard_notes.append("洪水浸水想定区域外（高台・非浸水）")
        
    # 液状化傾向
    liquefaction_geojson = fetch_mlit_geojson("XKT012", x, y, ZOOM)
    topography = "ローム台地"
    liquefaction_note = "液状化しにくい"
    if liquefaction_geojson.get("features"):
        for feat in liquefaction_geojson["features"]:
            poly = shape(feat["geometry"])
            if poly.contains(pt):
                props = feat.get("properties", {})
                topography = props.get("topographic_classification_name_ja", "ローム台地")
                liquefaction_note = props.get("note", "液状化しにくい")
                if "しやすい" in liquefaction_note:
                    hazard_score -= 12
                break
                
    # --- 3. 相場適正 ＆ 利便性（ベースライン加点） ---
    fair_value_score = 18
    livability_score = 9
    total_score = tranquility_score + hazard_score + fair_value_score + livability_score
    
    rank = "S" if total_score >= 90 else "A" if total_score >= 80 else "B" if total_score >= 70 else "C"
        
    return {
        "property_name": property_name,
        "lat": lat,
        "lon": lon,
        "total_score": total_score,
        "rank": rank,
        "tranquility": {"score": tranquility_score, "zone": detected_zone},
        "safety": {"score": hazard_score, "flood": flood_depth, "topography": topography, "liquefaction": liquefaction_note},
        "fair_value": {"score": fair_value_score},
        "livability": {"score": livability_score}
    }


## 5. Quiet Luxury 診断テキスト生成（LINE Bot / CRM 連携フォーマット）
誇張や煽り言葉を排除し、宅建士の冷静で洗練された視点で顧客に提示する診断カードテキストを自動生成します。

In [ ]:
def format_quiet_luxury_card(result: dict) -> str:
    """Sorai Tokyo 標準の Quiet Luxury 診断カードテキストを出力"""
    r = result
    card = f"""
🏛️ **Sorai Index 鑑定書 — {r['property_name']}**
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
**総合評価**: **{r['rank']} ランク** ({r['total_score']} / 100 点)

■ **静粛性・住環境 (Tranquility)**: {r['tranquility']['score']}/35 点
・指定地域: **{r['tranquility']['zone']}**
・都心の利便性を享受しつつ、落ち着いた住環境が守られたエリアです。

■ **安全性・地盤 (Safety & Resilience)**: {r['safety']['score']}/35 点
・浸水リスク: **{r['safety']['flood']}**
・地盤傾向: **{r['safety']['liquefaction']}**
・武蔵野台地の強固な地盤に位置し、自然災害リスクを最小限に抑えています。

■ **相場適正度 (Fair Value)**: **適正（お財布防衛水準）**
・近隣成約取引データに基づき、不当な付加価値分の上乗せがない適正な募集条件です。
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
※ 国土交通省「不動産情報ライブラリ」一次データに基づく客観評価
"""
    return card.strip()

# テスト実行（南青山エリア）
sample_eval = evaluate_sorai_index(35.663, 139.718, "南青山テラスレジデンス")
print(format_quiet_luxury_card(sample_eval))

## 6. インタラクティブ地図ビジュアライゼーション（Folium）
都心の主要高級レジデンスをプロットし、Sorai Index の評価ランク（S: ゴールド/ネイビー, A: ブルー）でピン留めしたインタラクティブマップを描画します。

In [ ]:
# 都心の代表的テスト物件リスト
BENCHMARK_PROPERTIES = [
    {"name": "広尾ガーデンヒルズ", "lat": 35.653, "lon": 139.721},
    {"name": "元麻布ヒルズ", "lat": 35.654, "lon": 139.731},
    {"name": "南青山テラスレジデンス", "lat": 35.663, "lon": 139.718},
    {"name": "神宮前レジデンス", "lat": 35.668, "lon": 139.710},
    {"name": "六本木タワーマンション", "lat": 35.662, "lon": 139.732},
]

# 地図の初期化（港区・渋谷区中心）
m = folium.Map(location=[35.658, 139.725], zoom_start=14, tiles="CartoDB positron")

# 物件を順次評価してプロット
results = []
for prop in BENCHMARK_PROPERTIES:
    res = evaluate_sorai_index(prop["lat"], prop["lon"], prop["name"])
    results.append(res)
    
    # ランクごとのピンカラー
    color = "darkblue" if res["rank"] == "S" else "blue" if res["rank"] == "A" else "gray"
    
    popup_html = f"""
    <div style='font-family: sans-serif; width: 220px;'>
        <h4 style='margin:0 0 5px 0; color:#1a2b4c;'>{res['property_name']}</h4>
        <p style='margin:0;'><b>Sorai Index:</b> <span style='font-size:16px; color:#d4af37;'><b>{res['rank']}ランク</b></span> ({res['total_score']}点)</p>
        <p style='margin:5px 0 0 0; font-size:12px; color:#555;'>
            ・用途: {res['tranquility']['zone']}<br>
            ・浸水想定: {res['safety']['flood']}<br>
            ・地盤: {res['safety']['liquefaction']}
        </p>
    </div>
    """
    
    folium.Marker(
        location=[prop["lat"], prop["lon"]],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{res['property_name']} ({res['rank']}ランク)",
        icon=folium.Icon(color=color, icon="home", prefix="fa")
    ).add_to(m)

print("✅ 5物件の評価が完了しました。地図を描画します。")
m

## 7. 評価データのエクスポート（BigQuery / CSV 連携）
計算結果を DataFrame に変換し、BigQuery または CSV / Parquet 形式で保存します。

In [ ]:
# DataFrame 化
df = pd.DataFrame([
    {
        "property_name": r["property_name"],
        "lat": r["lat"],
        "lon": r["lon"],
        "total_score": r["total_score"],
        "rank": r["rank"],
        "zoning": r["tranquility"]["zone"],
        "tranquility_score": r["tranquility"]["score"],
        "safety_score": r["safety"]["score"],
        "flood_depth": r["safety"]["flood"],
        "liquefaction": r["safety"]["liquefaction"],
        "fair_value_score": r["fair_value"]["score"],
    }
    for r in results
])

display(df)

# CSV 出力
output_csv = "sorai_index_benchmark.csv"
df.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"✅ 評価結果を {output_csv} にエクスポートしました。")